# Segmentación 3D usando CellPose

Este cuaderno demuestra un flujo de trabajo completo de segmentación 3D con aprendizaje profundo usando CellPose. Carga un volumen 3D con BioIO y Dask para eficiencia de memoria, lo visualiza en el visor interactivo de Napari, aplica Cellpose acelerado por GPU para segmentación celular 3D automatizada, y superpone las máscaras resultantes sobre el volumen original para su evaluación. Requiere bioio, napari y cellpose con soporte de GPU.

## Objetivos de aprendizaje

Al finalizar este cuaderno, serás capaz de:

1. **Cargar y visualizar datos de microscopía 3D** usando BioIO y Napari
    - Comprender cómo manejar eficientemente imágenes 3D grandes con arrays Dask
    - Navegar y renderizar volúmenes 3D en el visor de Napari

2. **Aplicar segmentación con aprendizaje profundo a datos 3D** usando Cellpose
    - Configurar Cellpose para tareas de segmentación 3D
    - Comprender los parámetros clave: `diameter`, `z_axis` y `do_3D`
    - Aprovechar la aceleración por GPU para un procesamiento más rápido

3. **Evaluar y refinar los resultados de segmentación**
    - Visualizar máscaras de segmentación superpuestas sobre los datos originales
    - Identificar cuándo los parámetros necesitan ajuste (p. ej., diámetro celular)
    - Reconocer escenarios que requieren reentrenamiento del modelo

4. **Desarrollar habilidades prácticas para flujos de trabajo de análisis de imágenes 3D**
    - Construir pipelines de análisis reproducibles en cuadernos Jupyter
    - Trabajar con datos reales de imágenes biológicas (conjunto de datos Lund)
    - Comprender la relación entre las propiedades de la imagen y los parámetros de segmentación

## Cargar los datos

Usar [BioIO](https://bioio-devs.github.io/bioio/OVERVIEW.html) con [Dask](https://docs.dask.org/en/stable/) nos permite transmitir el volumen 3D de forma diferida (lazy) para que se cargue sin agotar la memoria antes de enviarlo a Napari y Cellpose.

In [ ]:
from bioio import BioImage

In [2]:
image_handle = BioImage("../data/Lund.tif")

## Visualizar en Napari

Usaremos [Napari](https://napari.org/stable/) para visualizar interactivamente el volumen 3D.

In [3]:
import napari

# Usar Dask para carga diferida (lazy) de la imagen 3D pesada
# .dask_data de BioImage provee un array Dask sin cargar en memoria
image_data = image_handle.dask_data.squeeze()
image_data

dask.array<getitem, shape=(100, 256, 256), dtype=uint16, chunksize=(100, 256, 256), chunktype=numpy.ndarray>

In [4]:
# Crear un visor de Napari y agregar la imagen 3D
viewer = napari.Viewer()
viewer.add_image(image_data, name='Volumen Lund')
napari.run()

## Cargar y ejecutar CellPose en GPU

Cellpose es un modelo de aprendizaje profundo de código abierto para la segmentación general de células y núcleos en imágenes de microscopía. En este cuaderno, se usará para segmentar núcleos en el volumen 3D. Para más detalles, consultá la [documentación de Cellpose](https://cellpose.readthedocs.io/).

In [5]:
from cellpose import io, models, core

# Inicializar el modelo Cellpose con GPU habilitado
model = models.CellposeModel(gpu=True)

io.logger_setup() # ejecutar esto para obtener impresión del progreso

# Verificar si la instancia del cuaderno tiene acceso a GPU
if core.use_gpu()==False:
  raise ImportError("Sin acceso a GPU, cambiá tu entorno de ejecución")

2026-02-20 20:17:09,194 [INFO] WRITING LOG OUTPUT TO C:\Users\corba\.cellpose\run.log
2026-02-20 20:17:09,195 [INFO] 
cellpose version: 	4.0.8 
platform:       	win32 
python version: 	3.11.14 
torch version:  	2.10.0
2026-02-20 20:17:09,196 [INFO] ** TORCH CUDA version installed and working. **


In [6]:
# Ejecutar segmentación Cellpose en modo 3D
masks, flows, styles = model.eval(
    image_data.compute(),
    diameter=100,  # Ajustar según el tamaño típico de la célula en píxeles
    z_axis=0,  # Especificar el eje z para datos 3D
    do_3D=True  # Habilitar segmentación 3D
)

2026-02-20 20:17:09,546 [INFO] running YX: 100 planes of size (76, 76)
2026-02-20 20:17:17,358 [INFO] 100%|##########| 13/13 [00:07<00:00,  1.67it/s]
2026-02-20 20:17:17,376 [INFO] running ZY: 76 planes of size (100, 76)
2026-02-20 20:17:21,276 [INFO] 100%|##########| 10/10 [00:03<00:00,  2.57it/s]
2026-02-20 20:17:21,291 [INFO] running ZX: 76 planes of size (100, 76)
2026-02-20 20:17:25,189 [INFO] 100%|##########| 10/10 [00:03<00:00,  2.57it/s]
2026-02-20 20:17:25,212 [INFO] network run in 15.67s
2026-02-20 20:17:26,753 [INFO] masks created in 0.87s


In [7]:
viewer = napari.Viewer()
viewer.add_image(image_data, name="Volumen Lund", rendering="attenuated_mip")
viewer.add_labels(masks, name="Máscaras Cellpose", opacity=0.5)
viewer.dims.ndisplay = 3
napari.run()

2026-02-20 20:17:29,719 [INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


## Preguntas

1. ¿Hay algún parámetro que debería corregirse al ejecutar CellPose?

2. ¿Necesitamos reentrenar la red?

### Preguntas más difíciles

1. ¿Podés ejecutar Cellpose en el conjunto de datos llamado `lund1051_resampled.tif`? ¿Qué ocurre cuando elegís el tamaño correcto del núcleo celular?